In [27]:
# ==========================
# Import Required Libraries
# ==========================
 
import os
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
 
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage
from langchain_anthropic import ChatAnthropic

In [28]:

# ==========================
# Load Environment Variables
# ==========================
 
load_dotenv(override=True)
 
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
 
print("LLM Loaded Successfully!")

LLM Loaded Successfully!


In [29]:
#Step 5: Create Chat Node
def travel_bot(state: MessagesState):
 
    response = llm.invoke(state["messages"])
 
    return {
        "messages": [response]
    }

In [34]:
#Step 6: Build Graph
builder = StateGraph(MessagesState)
 
builder.add_node("travel_bot", travel_bot)
 
builder.add_edge(START, "travel_bot")
builder.add_edge("travel_bot", END)

In [35]:
#Step 7: Add Memory
#This is the main concept for today's task.
memory = MemorySaver()
 
graph = builder.compile(
    checkpointer=memory
)


In [36]:
#Step 8: Create Thread ID
#Very important.
#Same thread_id = same conversation memory.
config = {
    "configurable": {
        "thread_id": "trip001"
    }
}

In [39]:
#Step 9: Turn 1
result = graph.invoke(
    {
       "messages":[HumanMessage(content= "I am planning a trip to Bali.")]
    },
    config=config
)
 
print(result["messages"][-1].content)

# Bali Trip Planning

That sounds great! I'd love to help you plan. To give you the most useful advice, could you tell me a bit more:

**Trip Details:**
- How long are you planning to stay?
- When are you thinking of traveling?
- Is this your first time to Bali?

**Your Interests:**
- What appeals to you most? (beaches, culture, temples, hiking, nightlife, relaxation, food, adventure sports, etc.)
- Are you traveling solo, with a partner, or with family/friends?

**Budget & Style:**
- What's your approximate budget range?
- Do you prefer luxury, mid-range, or budget accommodations?

**Logistics:**
- Where are you traveling from?
- Do you have any visa concerns?

Once I know more about what you're looking for, I can give you tailored recommendations for:
- Where to stay
- What to see and do
- How to get around
- Where to eat
- Practical tips specific to your needs

What matters most to you for this trip?


In [40]:
#Step 10: Turn 2
result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="I prefer budget hotels.")
        ]
    },
    config=config
)
 
print(result["messages"][-1].content)

# Budget Bali Trip Planning

Great! Budget travel in Bali is very doable and popular. Here's what you should know:

## Budget Accommodation Options
- **Hostels:** $5-15/night - social atmosphere, often have common areas
- **Guesthouses/Homestays:** $10-25/night - local experience, basic but clean
- **Budget Hotels:** $15-30/night - private rooms, more comfort
- **Popular areas for budget stays:**
  - **Ubud** - backpacker hub, lots of budget options
  - **Canggu** - trendy but affordable hostels
  - **Seminyak** - mix of budget and mid-range
  - **Sanur** - quieter, good value

## Budget Travel Tips
- Eat at warungs (local restaurants) - meals $1-3
- Use public transport or share rides
- Many attractions are free or very cheap
- Book accommodations directly or through budget sites

## Money-Saving Suggestions
- Stay in Ubud - cheapest area with lots to do
- Travel during shoulder season (May-June, Sept-Oct)
- Get a scooter rental ($3-5/day) instead of taxis

## Still Helpful to Know:
-

In [41]:
#Step 11: Turn 3
result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="My trip is in December.")
        ]
    },
    config=config
)
 
print(result["messages"][-1].content)

# Bali in December - Budget Trip

Good to know! Here are important things about December travel:

## Weather & Season
- **Wet/Rainy Season** - expect afternoon showers and humidity
- **Pros:** Fewer tourists, lower prices on accommodations and flights
- **Cons:** Some beach days disrupted, occasional flooding, higher humidity

## Budget Advantages in December
- Hotel rates drop significantly (20-40% cheaper than peak season)
- Flights may be cheaper (book early though)
- Restaurants and attractions less crowded
- Better negotiating power on scooter rentals

## December Considerations
- **Christmas/New Year period** (Dec 20-Jan 5) - prices spike, book early
- **Early December** (1-15) - best value and fewer crowds
- Pack light rain jacket or poncho
- Some outdoor activities may be limited

## Still Helpful:
To give you the best budget recommendations, could you tell me:
- **Early December or late December?** (affects pricing significantly)
- **How many days** are you staying?
- **Solo o

In [42]:
#Step 12: Turn 4
result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="Summarise my travel plan.")
        ]
    },
    config=config
)
 
print(result["messages"][-1].content)

# Your Bali Travel Plan Summary

**Destination:** Bali, Indonesia

**Travel Time:** December

**Budget Level:** Budget accommodations ($10-30/night)

**Accommodation Style:** 
- Hostels, guesthouses, or budget hotels
- Likely in areas like Ubud, Canggu, or Seminyak

**Season:** Wet/rainy season - fewer tourists, lower prices

---

## What We Still Need to Finalize:

To complete your plan, I'd need to know:
- **Exact dates** (early or late December?)
- **Trip duration** (how many days/nights?)
- **Travel companions** (solo, partner, group?)
- **Main interests** (beaches, culture, temples, hiking, nightlife, food, etc.)

Once you provide these details, I can give you:
- Specific accommodation recommendations
- Day-by-day itinerary
- Budget breakdown
- Best areas to stay
- Activities and attractions to prioritize

Would you like to share any of these details?


                START
                  │
                  ▼
        ┌─────────────────┐
        │   travel_bot    │
        │  (LLM Node)     │
        └─────────────────┘
                  │
                  ▼
                 END
 
MemorySaver
      ▲
      │
thread_id = "trip001"
      │
Stores conversation across all turns
 